# PanVision — preprocess PANORAMA into Google Drive (run before the training notebooks)

Uses the team's preprocessing code unchanged (`src/preprocessing`): HU window −100..300 → 0..1, 1.5×1.5×3 mm,
LPS. Each CT is downloaded from the public HuggingFace mirror to Colab's local disk, processed, copied to
`processed/full/<study_id>/` in the shared Drive folder (~10 MB each), and the raw CT (~70 MB) is deleted.

**Runtime → Change runtime type → CPU** (no GPU needed; keeps your GPU quota for training).
If the session disconnects, run all cells again: finished studies are skipped.

In [ ]:
import os
%cd /content
# Leave the repo folder before re-cloning it, so re-running this cell is safe.
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/Panvision && git clone -q -b segmentation https://{token}@github.com/hrihaanG/Panvision.git /content/Panvision
!git -C /content/Panvision remote set-url origin https://github.com/hrihaanG/Panvision.git
%cd /content/Panvision
assert os.path.exists('src/segmentation'), 'Clone failed: check the GITHUB_TOKEN secret and its notebook access'
!pip -q install SimpleITK openpyxl huggingface_hub

In [ ]:
import os, glob
# Google Drive folder that receives everything (raw/annotations, processed/full). Change if yours differs.
DATA_FOLDER = 'PanVision-OneDrive'
hits = sorted(glob.glob(f'/content/drive/MyDrive/{DATA_FOLDER}') + glob.glob(f'/content/drive/MyDrive/*/{DATA_FOLDER}'))
assert hits, f'{DATA_FOLDER} not found in My Drive (shared with you? add it as a shortcut in My Drive)'
# Always re-point the link: an earlier run in this runtime may have linked a different folder.
if os.path.islink('/content/data'):
    os.unlink('/content/data')
os.symlink(hits[0], '/content/data')  # space-free link
assert os.path.basename(os.path.realpath('/content/data')) == DATA_FOLDER
print('Using', os.path.realpath('/content/data'))
# Official annotations: public, ~80 MB.
!rm -rf /content/panorama_labels && git clone -q --depth 1 https://github.com/DIAGNijmegen/panorama_labels.git /content/panorama_labels
!ls /content/panorama_labels
# One-off copy into Drive so the training notebooks (which look for raw/annotations) find this folder.
if not os.path.exists('/content/data/raw/annotations/clinical_information.xlsx'):
    !mkdir -p /content/data/raw && rsync -a --exclude .git /content/panorama_labels/ /content/data/raw/annotations/


## Settings — the only cell you edit
Full cohort (~2,200 studies, ~22 GB): split across teammates. Everyone uses the same `NUM_SHARDS`, each person a different `SHARD` (0, 1, 2, …), each in their **own** Google account. Outputs land in the same shared Drive folder and count against the Drive of whoever runs the shard.

Set `FULL = False` for the balanced 460-study subset (one person, ~1–2 h).

In [ ]:
FULL = True
NUM_SHARDS = 4
SHARD = 0   # <- each teammate changes this: 0, 1, 2, 3
SELECTION = '--all' if FULL else '--train 300 --val 80 --test 80'

## Plan (no downloads)

In [ ]:
!python -m src.segmentation.prepare_subset --shared /content/data --annotations /content/panorama_labels {SELECTION} --shard {SHARD} --num-shards {NUM_SHARDS} --workers 4 --dry-run

## Run (re-run after a disconnect to continue)

In [ ]:
!python -m src.segmentation.prepare_subset --shared /content/data --annotations /content/panorama_labels {SELECTION} --shard {SHARD} --num-shards {NUM_SHARDS} --workers 4